# Konu 10 — Zincirleme: üretim hattı

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Konu 5'te slogan yazdırdın, Konu 8'de görselleri koddan ve toplu olarak ürettin; her biri ayrı bir defterdi. Bugün ikisini
tek bir **hatta** bağlıyoruz: `ayarlar.json`'daki brif → metin modeliyle sloganlar → görsel modeliyle her slogana bir görsel.
Bir halkanın çıktısı, sıradaki halkanın girdisi.

Isınma, Adım 1 ve Adım 2 anahtarsız. Adım 3'ten sonrası `anahtar.txt` ister. Defterin tamamı 4 metin ve 3 görsel isteği
gönderir (Bonus 2 ile 4 görsel); Adım 6'da bilerek reddedilen 3 görsel isteği de var. Hepsi ücretsiz günlük kotanın küçük bir kısmı.

## Isınma — Hat kâğıt üstünde

Önce sahte halkalarla: üç küçük fonksiyon, hiçbiri internete çıkmıyor. Bu hücre bir şey yazmaz, yalnızca
fonksiyonları tanımlar.

In [ ]:
def slogan_yaz(konu):
    return konu + ": burada zaman yavaşlar"

def yonerge_kur(slogan, stil):
    return stil + " poster, idea: " + slogan

def gorsel_ciz(yonerge):
    return "[görsel] " + yonerge

Şimdi zinciri kuruyoruz: her satır bir öncekinin sonucunu alıyor.

In [ ]:
konu = "sessiz bir çalışma kafesi"
slogan = slogan_yaz(konu)
yonerge = yonerge_kur(slogan, "risograph")
gorsel = gorsel_ciz(yonerge)

print(slogan)
print(yonerge)
print(gorsel)

**Tahmin et, sohbete yaz:** ilk satırdaki konuyu `"gece açık bir kitapçı"` yapıp yeniden çalıştırırsan üç satırdan kaçı değişir?

Aşağıdaki hücre üsttekinin kopyası; yalnızca ilk satırı farklı. Çalıştır ve tahminini kontrol et.

In [ ]:
konu = "gece açık bir kitapçı"
slogan = slogan_yaz(konu)
yonerge = yonerge_kur(slogan, "risograph")
gorsel = gorsel_ciz(yonerge)

print(slogan)
print(yonerge)
print(gorsel)

Üç satırın üçü de değişti. Hattın başındaki tek girdi sonuna kadar taşınıyor: başta yanlış bir kelime, sonda yanlış bir görsel.
Bu yüzden hattın en önemli satırı brif. Gerçek hatta her halka bir istek; aşağıda halkaları gerçek modellerle değiştireceğiz.

## Adım 1 — Ayarlar dosyası: `ayarlar.json`

Bu klasörde `ayarlar.json` diye bir dosya var. VS Code'da aç: hattın bütün ayarları orada, koddan ayrı. Bu bir **yapılandırma
dosyası** (config): müşteri ya da konu değişince kodu değil bu dosyayı değiştiriyoruz.

Biçimi JSON: tek katmanlı bir sözlük. Konu 8'de günlüğü `json.dump` ile dosyaya yazdın; `json.load` bunun tersi: dosyayı okur,
Python sözlüğüne çevirir (Konu 9'da video günlüğünü böyle geri okudun).

In [ ]:
import json

with open("ayarlar.json", encoding="utf-8") as dosya:
    ayarlar = json.load(dosya)

print(ayarlar)

Her ayarı bir değişkene alıyoruz; aşağıdaki hücreler bu kısa adları kullanacak:

In [ ]:
calisma = ayarlar["calisma"]
konu = ayarlar["konu"]
stil = ayarlar["stil"]
slogan_sayisi = ayarlar["slogan_sayisi"]
sicaklik = ayarlar["sicaklik"]
tohum = ayarlar["tohum"]

print(konu, "/", stil)

`konu` Türkçe (Türkçe slogan sorusuna gidecek), `stil` İngilizce (görsel yönergesine gidecek; Konu 7).
`ayarlar.json`'u değiştirirsen bu iki hücreyi yeniden çalıştır; yoksa değişkenler eski değerleri taşır.

## Adım 2 — Çalışma klasörü

Hattın bütün çıktıları tek bir klasöre gidecek: `ciktilar/` içinde, adı `ayarlar.json`'daki `calisma` alanından gelen bir klasör.

- Yol, Konu 8'deki `klasor = "izgara/"` gibi düz bir metin: `"ciktilar/"` + çalışma adı + `"/"`. Sondaki `/` sayesinde
  dosya adını doğrudan ekleyebiliriz: `klasor + "sloganlar.json"`.
- `os.makedirs(klasor, exist_ok=True)` klasörü açar; `ciktilar/` henüz yoksa onu da. `exist_ok=True`: klasör zaten varsa
  hata verme. `os` Python'un içinde geliyor; Konu 8'deki `os.path.exists` da onun.
- Konu 8'de klasörü `sabirli_ciz` kendisi açıyordu. Bu hatta ilk yazılan dosya bir görsel değil, sloganlar olacak; o yüzden
  klasörü baştan biz açıyoruz.

In [ ]:
import os

klasor = "ciktilar/" + calisma + "/"
os.makedirs(klasor, exist_ok=True)
print(klasor)

Klasörün adına neden otomatik bir tarih eklemedik? Her çalıştırmada yeni bir klasör açılırdı; içinde hiçbir şey "zaten var"
olmazdı ve hat her şeyi baştan üretirdi. Kural Konu 8'deki gibi: **aynı iş, aynı ad**. Brifi değiştirdiğinde (konu, stil, tohum)
`calisma` adını da sen değiştir; yeni brif yeni klasörde başlar, eskisi olduğu gibi kalır.

## Adım 3 — Hazır halkalar

`halkalar.py` bu klasördeki yardımcı dosya. Önceki konulardan iki fonksiyon içinde hazır:

- `modele_sor(soru, sicaklik)`: Konu 5'te yazdığın fonksiyon. Tek farkı: istek başarısız olursa çökmez, `None` verir.
- `sabirli_ciz(yonerge, tohum, yol)`: Konu 8'deki fonksiyon. Görseli en fazla üç kez dener ve `yol`a kendisi kaydeder;
  başarılıysa `yol`u, olmazsa `None` verir.

`halkalar.py` anahtarı `../anahtar.txt`'den kendisi okur. Halkalar hazır; onları **birbirine bağlamak** bugün senin işin.

Aşağıdaki hücre bir şey yazmaz. `FileNotFoundError: ... '../anahtar.txt'` görürsen anahtar dosyası yok ya da yanlış yerde:
`gita3111` klasöründe durmalı (Konu 2).

In [ ]:
from halkalar import modele_sor, sabirli_ciz

Konu 5'teki slogan sorusunu bir kalıba çeviriyoruz: sabit kısım ve sonunda brifteki konu. Önce tek bir istek.

**Çalıştırmadan önce tahmin et:** model yalnızca sloganı mı yazacak, yoksa başına sonuna bir şey ekleyecek mi?

In [ ]:
soru = "Şu konu için tek bir kısa slogan yaz. Yalnızca sloganı yaz, açıklama ekleme: " + konu

cevap = modele_sor(soru, sicaklik)
print(cevap)

Model talimata her zaman uymaz: sloganı tırnak içine alabilir, başına "Slogan:" yazabilir, sonuna bir açıklama ekleyebilir.
Bu cevap bir sonraki halkaya (görsel yönergesine) gidecek; önce **temizliyoruz**. İki tanıdık araç:
`.strip()` baştaki ve sondaki boşlukları siler (Konu 2'de anahtar dosyasını okurken), `.replace('"', "")` düz tırnakları
siler (Konu 1'de metni temizlerken).

Üstteki hücre cevap yerine `İstek başarısız, durum kodu: 401` yazdıysa anahtar yanlış ve `cevap` `None`; bu hücre
`AttributeError: 'NoneType' object has no attribute 'strip'` verir. `anahtar.txt`'yi düzelt, çekirdeği yeniden başlat
(**Restart**), yukarıdan yeniden çalıştır.

In [ ]:
temiz = cevap.strip()
temiz = temiz.replace('"', "")
print(temiz)

Temiz cevapta hâlâ bir fazlalık varsa (kıvrık tırnak “ ”, başta `Slogan:`) aynı kalıpla bir satır daha ekle:
`temiz = temiz.replace("Slogan: ", "")`. Adım 4'teki fonksiyonda aynı iki temizleme satırı var; eklediğin satırı oraya da ekle
(orada değişkenin adı `slogan`).

## Adım 4 — 1. halka: sloganlar (ara çıktıyı sakla)

Bir halkanın sonucu, hattın **ara çıktısı**. Sloganları bir dosyaya yazıp saklayacağız: `sloganlar.json`.
Fonksiyon her çağrıldığında önce sorar: **zaten var mı?**

- Dosya varsa (`os.path.exists`, Konu 8): oku, ver; modele hiç istek gitmez (`return` fonksiyonu orada bitirir).
- Yoksa: `slogan_sayisi` kez sor (`range(slogan_sayisi)`: `slogan_sayisi` 3, yani "3 kez"), her cevabı temizleyip listeye
  ekle, listeyi dosyaya yaz, ver.
- Bir cevap gelmezse (`None`) o turu atla (`continue`); hat durmaz.

Dosyaya yazmak Konu 8'deki günlükle aynı: `json.dump(..., ensure_ascii=False, indent=2)`.

Fonksiyon parametre almıyor; `soru`, `sicaklik` ve `slogan_sayisi`'yı yukarıdaki hücrelerden kullanıyor. Hücre yalnızca
fonksiyonu tanımlar, bir şey yazmaz; istekler fonksiyonu çağırınca, bir sonraki hücrede gider. Adım 3'teki deneme sloganı
listeye girmez: fonksiyon üç yeni istek gönderir.

In [ ]:
slogan_dosyasi = klasor + "sloganlar.json"

def slogan_halkasi():
    if os.path.exists(slogan_dosyasi):
        with open(slogan_dosyasi, encoding="utf-8") as dosya:
            sloganlar = json.load(dosya)
        print("Dosyadan okundu, istek gitmedi")
        return sloganlar
    sloganlar = []
    for tur in range(slogan_sayisi):
        cevap = modele_sor(soru, sicaklik)
        if cevap is None:
            print("Slogan gelmedi, atlıyorum")
            continue
        slogan = cevap.strip()
        slogan = slogan.replace('"', "")
        sloganlar.append(slogan)
    with open(slogan_dosyasi, "w", encoding="utf-8") as dosya:
        json.dump(sloganlar, dosya, ensure_ascii=False, indent=2)
    print("Üretildi ve kaydedildi")
    return sloganlar

In [ ]:
sloganlar = slogan_halkasi()
print(sloganlar)

`ciktilar/kafe_afisi_1/sloganlar.json`'u VS Code'da aç: üç slogan orada.

Üç kez `Slogan gelmedi` ve boş bir liste (`[]`) gördüysen istekler başarısız oldu ve dosyaya boş liste yazıldı. Sorunu
(çoğunlukla anahtar) düzelttikten sonra `sloganlar.json`'u sil; yoksa hat bu boş listeyle devam eder.

**Tahmin et, sohbete yaz:** aynı çağrıyı bir kez daha yaparsak kaç istek gider? Sloganlar aynı mı kalır?

In [ ]:
sloganlar = slogan_halkasi()
print(sloganlar)

Yalnızca kota için değil, **tutarlılık** için de saklıyoruz. Sıcaklık 0.9; her istekte başka bir slogan gelir (Konu 5). Sloganları her
çalıştırmada yeniden üretseydik `gorsel_1.jpg` dünkü sloganla üretilmiş olurdu, günlükte ise bugünkü slogan yazardı: görsel ile
metin birbirini tutmazdı.

Kural: **pahalı ya da her seferinde farklı çıkan** ara çıktıyı sakla.

## Adım 5 — 2. halka: her slogana bir görsel

Önce her slogan için görsel yönergesini kuruyoruz: stil (İngilizce, ayarlardan) + slogan + "görselde yazı olmasın".
Bu bir kalıp: sabit kısım ve iki değişen yer (stil ve slogan). Yönergeleri saklamıyoruz: sloganlardan ve ayarlardan her
seferinde aynı biçimde, bedavaya kurulur.

In [ ]:
yonergeler = []
for slogan in sloganlar:
    yonerge = stil + ", a poster illustration that expresses this idea: " + slogan + ", no text, no letters"
    yonergeler.append(yonerge)
    print(yonerge)

Sloganlar Türkçe, yönergenin geri kalanı İngilizce. Görsel model çoğunlukla İngilizce metinle eğitildi (Konu 7); Türkçe
sloganın **fikrini** ne kadar anlayacak, birazdan göreceğiz. "Görselde yazı olmasın" bilerek orada: görsel modeller harfleri
çoğu zaman bozar; sloganı afişe tasarımcı dizer. Hat **taslak** üretir.

Görsel halkası tek bir görsel için çalışan bir fonksiyon. Slogan halkasıyla aynı soruyu sorar: zaten var mı? Varsa atla
(`os.path.exists(yol)`; dosya orada olduğu için kayıtta durum yine `uretildi`), yoksa `sabirli_ciz` ile üret; görseli dosyaya
o kaydeder. Her seferinde bir **kayıt**
(sözlük) verir: hangi halka, ne girdi, hangi dosya çıktı, sonuç ne (`durum`).

- `"gorsel_" + str(sira) + ".jpg"`: `str` sayıyı metne çevirir (Konu 5); metne yalnızca metin eklenebilir.
- `return kayit` fonksiyonu orada bitirir; hangi yoldan çıkarsa çıksın elimize bir kayıt gelir.

In [ ]:
def gorsel_halkasi(sira, yonerge):
    yol = klasor + "gorsel_" + str(sira) + ".jpg"
    kayit = {"halka": "gorsel", "girdi": yonerge, "cikti": yol}
    if os.path.exists(yol):
        kayit["durum"] = "uretildi"
        print("Zaten var, atlıyorum:", yol)
        return kayit
    sonuc = sabirli_ciz(yonerge, tohum, yol)
    if sonuc is None:
        kayit["durum"] = "olmadi"
        print("Olmadı, atlıyorum:", yol)
        return kayit
    kayit["durum"] = "uretildi"
    print("Kaydedildi:", yol)
    return kayit

Halkayı bir döngüyle her yönergeye uyguluyoruz; gelen kayıt listeye ekleniyor. `sira` her turun **başında**
bir artıyor: ilk görsel `gorsel_1.jpg`. Üç görsel bir dakika kadar sürer.

In [ ]:
kayitlar = []
sira = 0
for yonerge in yonergeler:
    sira = sira + 1
    kayit = gorsel_halkasi(sira, yonerge)
    kayitlar.append(kayit)

print(len(kayitlar), "kayıt")

`ciktilar/kafe_afisi_1/` klasöründeki görsellere bak. Sohbete yaz:

- Görselde yine de yazı çıktı mı? Harfler okunuyor mu?
- Görseller sloganların **fikrini** mi taşıyor, yoksa yalnızca stili mi? Üç görsel birbirinden ne kadar farklı?

## Adım 6 — Bir halka koparsa hat durmasın

Gerçek bir hatta bir halka er geç kopar: internet gider, kota biter, bir yönerge reddedilir. Derste bunu beklemek yerine bilerek
koparıyoruz: dördüncü görseli **boş bir yönergeyle** istiyoruz. Görsel model boş yönergeyi kabul etmez.

**Tahmin et:** hücre hata verip duracak mı? Kayıtta ne yazacak? (Üç denemeyle 10 saniye kadar sürer.)

In [ ]:
kayit = gorsel_halkasi(4, "")
kayitlar.append(kayit)

print(kayit)
print(len(kayitlar), "kayıt")

Sunucu üç kez de reddetti (durum kodu 400). Ama hücre çökmedi: `gorsel_halkasi` hata yerine `"durum": "olmadi"` yazan bir
kayıt verdi. Bir döngünün içinde olsaydık döngü sıradaki görsele geçerdi. Hat iki şeyi birlikte yapıyor:

1. Kopan halkayı **kayda geçiriyor** (hangi görsel, `durum`: `olmadi`).
2. Kopmayanlarla **devam ediyor**.

`gorsel_4.jpg` hiç oluşmadı; hattı yarın yeniden çalıştırırsan yalnızca o denenir.

Bu hücreyi iki kez çalıştırırsan aynı kayıt listeye iki kez girer (`5 kayıt`). Listeyi düzeltmek için Adım 5'in döngü
hücresinden yeniden başla: görseller zaten var, istek gitmez.

## Adım 7 — Üretim günlüğü: `uretim.json`

Hattın bütün kayıtları tek dosyada, **üretim günlüğünde**: hangi ayarlarla çalıştı, hangi sloganlar çıktı, her görselin sonucu
ne. Konu 8'deki günlüğün aynısı; bu kez bir sözlüğün içinde üç bölüm.

In [ ]:
gunluk = {"ayarlar": ayarlar, "sloganlar": sloganlar, "kayitlar": kayitlar}

with open(klasor + "uretim.json", "w", encoding="utf-8") as dosya:
    json.dump(gunluk, dosya, ensure_ascii=False, indent=2)

print("Günlük yazıldı:", len(kayitlar), "kayıt")

`ciktilar/kafe_afisi_1/uretim.json`'u VS Code'da aç. `durum`u `olmadi` olan kayıtlar eksiklerin listesi: hattı yeniden
çalıştırınca denenecek olanlar. Final projesinin teslimindeki **üretim günlüğü** bu dosya: hangi brif, hangi model ayarı,
ne çıktı, ne eksik kaldı.

**Defteri yarın baştan çalıştırırsan ne görürsün?** Adım 3 yine bir deneme isteği gönderir. Adım 4'te iki çağrı da
`Dosyadan okundu, istek gitmedi` yazar; Adım 5 de üç görsel için `Zaten var, atlıyorum` yazar. Adım 6 yine üç kez reddedilir:
`gorsel_4.jpg` hiç oluşmadığı için her seferinde yeniden denenir. Reddedilenler dışında giden tek istek Adım 3'teki deneme.
Hat kaldığı yerden devam ediyor: biten iş yeniden üretilmiyor, yalnızca eksik olan deneniyor.

## Bonus

**1. Pano: slogan ve görsel yan yana.** Konu 8'deki kontak sayfasının tek satırlık hâli: her kutuda bir görsel, üstünde onu
doğuran slogan. `plt.imread` görseli okur, `plt.imshow` kutuya koyar; dosya yoksa kutu boş kalır.

In [ ]:
import matplotlib.pyplot as plt

sutun = len(sloganlar)

plt.figure(figsize=(12, 5))
sira = 0
for slogan in sloganlar:
    sira = sira + 1
    yol = klasor + "gorsel_" + str(sira) + ".jpg"
    plt.subplot(1, sutun, sira)
    if os.path.exists(yol):
        resim = plt.imread(yol)
        plt.imshow(resim)
    plt.title(slogan, fontsize=9)
    plt.axis("off")

plt.savefig(klasor + "pano.png", bbox_inches="tight")
plt.show()

Panoya tasarımcı gözüyle bak. Hangi slogan hangi görselle güçlü? Slogan ile görselin **aynı şeyi söylediği** kutu mu daha iyi,
yoksa birbirini **tamamladığı** kutu mu (görsel bir şey gösteriyor, slogan ona yeni bir anlam ekliyor)? Sohbete tek cümle.

**2. Ara çıktıya elle dokun.** Ara çıktıyı saklamanın bir faydası daha var: hattın ortasında araya girebilirsin.

1. `ciktilar/kafe_afisi_1/sloganlar.json`'u VS Code'da aç, ikinci sloganı kendi yazdığın bir sloganla değiştir, kaydet.
   (Tırnakları ve virgülleri bırak; JSON biçimi bozulmasın.)
2. Aynı klasördeki `gorsel_2.jpg`'yi sil.
3. Adım 4'ün ilk çağrı hücresinden (`sloganlar = slogan_halkasi()`) Adım 5'in sonuna kadar sırayla yeniden çalıştır,
   sonra Adım 7'nin günlük hücresini ve panoyu.

Kaç istek gitti? Pano neden yalnızca bir kutuda değişti?

## Sıradaki konu

Bu hatta adımların sırasını sen belirledin. Konu 11'de modele bilmediği bir belgeyi okutacağız, Konu 12'de sırayı model belirleyecek.